In [1]:
import numpy as np
import pandas as pd
import json

# Data loading into a Pandas Data Frame

In [2]:
ls ../../data

csv/                              luminaries650.json
delete_luminaries.sh*             luminaries-trafficLights-100.json
delete_PoIs.sh*                   luminaries-trafficLights-10.json
delete_traffic_lights.sh*         luminaries-trafficLights-650.json
insert_luminaries_Madrid.sh*      POI/
insert_PoIs.sh*                   poi100.json
insert_traffic_lights_Madrid.sh*  poi10.json
list_luminaries.sh*               poi650.json
list_PoIs.sh*                     scripts.txt
list_traffic_lights.sh*           traffic_lights/
luminaries/                       trafficLights100.json
luminaries100.json                trafficLights10.json
luminaries10.json                 trafficLights650.json


In [20]:
FILE_LUM = "../../data/luminaries650.json"
FILE_TL = "../../data/trafficLights650.json"

df_lum = pd.read_json(FILE_LUM)
df_tl = pd.read_json(FILE_TL)

# split coordinates into lon / lat
df_lum[["lon", "lat"]] = pd.DataFrame(
    df_lum["location"].apply(lambda x: x["coordinates"]).to_list(),
    index=df_lum.index
)
df_tl[["lon", "lat"]] = pd.DataFrame(
    df_tl["location"].apply(lambda x: x["coordinates"]).to_list(),
    index=df_tl.index
)

# optional: drop the nested location column
df_lum = df_lum.drop(columns=["location"])
df_tl = df_tl.drop(columns=["location"])


In [21]:
df_lum.head()

,id,type,district,neighborhood,status,people_count,lon,lat
0,urn:ngsi-ld:Luminaries:148,Luminaries,16,4,on,1,-3.643609,40.473651
1,urn:ngsi-ld:Luminaries:149,Luminaries,16,4,on,1,-3.643704,40.473522
2,urn:ngsi-ld:Luminaries:150,Luminaries,16,4,on,1,-3.643525,40.473762
3,urn:ngsi-ld:Luminaries:230000,Luminaries,13,4,broken,4,-3.650733,40.381050
4,urn:ngsi-ld:Luminaries:230001,Luminaries,13,4,on,0,-3.648982,40.380154


In [22]:
df_tl.head(10)

,id,type,district,description,installationDate,status,lon,lat
0,urn:ngsi-ld:TrafficLightSignal:100,TrafficLightSignal,9,PRINCESA - SAN LEONARDO - PL. DE LOS CUBOS,27/02/1962,red,-3.712023,40.424761
1,urn:ngsi-ld:TrafficLightSignal:10027,TrafficLightSignal,15,CONDADO DE TREVIÑO - CALERUEGA,28/12/2020,green,-3.670974,40.479511
2,urn:ngsi-ld:TrafficLightSignal:1004,TrafficLightSignal,15,TORRELAGUNA - ARTURO BALDASANO - P.P.,15/04/1981,red,-3.660855,40.455650
3,urn:ngsi-ld:TrafficLightSignal:1005,TrafficLightSignal,10,GALLUR 435 P.P.,29/04/1981,green,-3.741989,40.396482
4,urn:ngsi-ld:TrafficLightSignal:101,TrafficLightSignal,9,PRINCESA - ROMERO ROBLEDO,16/06/1962,red,-3.718005,40.433007
5,urn:ngsi-ld:TrafficLightSignal:102,TrafficLightSignal,7,SAN BERNARDO - MAGALLANES,31/07/1962,red,-3.705185,40.431425
6,urn:ngsi-ld:TrafficLightSignal:1020,TrafficLightSignal,7,BLASCO DE GARAY - DONOSO CORTES,08/09/1981,green,-3.711339,40.436313
7,urn:ngsi-ld:TrafficLightSignal:1021,TrafficLightSignal,7,FERNANDO CATOLICO - ARCIPRESTE HITA,08/09/1981,yellow,-3.718065,40.434301
8,urn:ngsi-ld:TrafficLightSignal:1029,TrafficLightSignal,10,CEBREROS - CARLINA - ACC. A-5,01/10/1981,yellow,-3.750700,40.403876
9,urn:ngsi-ld:TrafficLightSignal:103,TrafficLightSignal,7,AV. FILIPINAS - VALLEHERMOSO - LUCIO DEL VALLE,04/09/1962,yellow,-3.708133,40.440789


## Merge both datasets (cartesian product) to filter elements that are close (ex. 100 mts)

In [23]:
import numpy as np
import pandas as pd


def cartesian_haversine(
    df_luminaires: pd.DataFrame,
    df_trafficlights: pd.DataFrame,
    *,
    lon_col: str = "lon",
    lat_col: str = "lat",
    suffixes=("_lum", "_tl"),
    distance_col: str = "distance",
    unit: str = "m",
) -> pd.DataFrame:
    """
    Create a cartesian product of two DataFrames and compute Haversine distance
    between their geographic coordinates.

    Parameters
    ----------
    df_luminaires, df_trafficlights : pd.DataFrame
        Input DataFrames containing longitude/latitude columns.
    lon_col, lat_col : str
        Column names for longitude and latitude (must exist in both DataFrames).
    suffixes : tuple
        Suffixes applied to overlapping column names.
    distance_col : str
        Name of the output distance column.
    unit : {"m", "km"}
        Unit of distance (meters or kilometers).

    Returns
    -------
    pd.DataFrame
        Cartesian-merged DataFrame with a distance column.
    """

    if unit not in {"m", "km"}:
        raise ValueError("unit must be 'm' or 'km'")

    # Earth radius
    R = 6371000.0 if unit == "m" else 6371.0

    # Cartesian product
    df = df_luminaires.merge(df_trafficlights, how="cross", suffixes=suffixes)

    # Extract and convert to radians
    lat1 = np.radians(df[f"{lat_col}{suffixes[0]}"].to_numpy())
    lon1 = np.radians(df[f"{lon_col}{suffixes[0]}"].to_numpy())
    lat2 = np.radians(df[f"{lat_col}{suffixes[1]}"].to_numpy())
    lon2 = np.radians(df[f"{lon_col}{suffixes[1]}"].to_numpy())

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    df[distance_col] = R * c

    return df


In [24]:
lum_tl_df = (
    cartesian_haversine(
        df_lum,
        df_tl,
        suffixes=("_lum", "_tl"),
        unit="m",
    )
    .sort_values("distance", ascending=True)
)
lum_tl_df.head(4)

,id_lum,type_lum,district_lum,neighborhood,status_lum,people_count,lon_lum,lat_lum,id_tl,type_tl,district_tl,description,installationDate,status_tl,lon_tl,lat_tl,distance
216842,urn:ngsi-ld:Luminaries:230330,Luminaries,13,5,on,2,-3.645282,40.389427,urn:ngsi-ld:TrafficLightSignal:363,TrafficLightSignal,13,AV. DE LA ALBUFERA - SANTIAGO ALIO,01/07/1969,green,-3.645161,40.389523,14.788671
247836,urn:ngsi-ld:Luminaries:230378,Luminaries,15,9,on,1,-3.665118,40.478452,urn:ngsi-ld:TrafficLightSignal:1675,TrafficLightSignal,15,GOLFO DE SALONICA - DALIA,27/04/2001,green,-3.664924,40.478393,17.676374
175837,urn:ngsi-ld:Luminaries:230267,Luminaries,13,5,on,1,-3.646187,40.390186,urn:ngsi-ld:TrafficLightSignal:236,TrafficLightSignal,13,AV. DE LA ALBUFERA - PEDRO LABORDE,13/07/1967,green,-3.646442,40.390315,25.948386
178437,urn:ngsi-ld:Luminaries:230271,Luminaries,13,5,on,1,-3.646736,40.390400,urn:ngsi-ld:TrafficLightSignal:236,TrafficLightSignal,13,AV. DE LA ALBUFERA - PEDRO LABORDE,13/07/1967,green,-3.646442,40.390315,26.636090


In [25]:
df_llm_out10 = pd.read_csv("llm_output-650.csv")
df_llm_out10.head(3)

,QN,short_name,question,chatGPT4.1-t0.0,Llama3.1-8B-instruct,Qwen3-8B-Base,deepseek-llm-7b-base,smolLM-1.7B,ground_truth
0,Q1,TL_Status_Specific,What is the current status of the traffic ligh...,TORRELAGUNA - ARTURO BALDASANO - P.P.,The current status of the traffic light at the...,The current status of the traffic light at the...,<EMPTY_ANSWER>,NaN,NaN
1,Q2,TL_Filter_Red,Please show me a list of all traffic lights th...,Here are the traffic lights that are currently...,Here are the traffic lights that are currently...,<EMPTY_ANSWER>,<EMPTY_ANSWER>,NaN,NaN
2,Q3,TL_Filter_Yellow,Identify all traffic lights currently in the '...,Here are the traffic lights currently in the '...,The traffic lights currently in the 'Yellow' s...,<EMPTY_ANSWER>,<EMPTY_ANSWER>,NaN,NaN


# Q1: What is the current status of the traffic light at the intersection of TORRELAGUNA y ARTURO BALDASANO?

For this question, I need to inspect TL dataset and check the lights at the mentioned intersection
It is straightforward to see that the status of the light is "red".

In [26]:
df_tl.iloc[2]

id                    urn:ngsi-ld:TrafficLightSignal:1004
type                                   TrafficLightSignal
district                                               15
description         TORRELAGUNA - ARTURO BALDASANO - P.P.
installationDate                               15/04/1981
status                                                red
lon                                             -3.660855
lat                                              40.45565
Name: 2, dtype: object

# Q2: Please show me a list of all traffic lights that are currently 'Red'

In [27]:
q2_df = df_tl[df_tl['status']=="red"]
print(f"There are a total of {len(q2_df)} lights in red status")
q2_df

There are a total of 287 lights in red status


,id,type,district,description,installationDate,status,lon,lat
0,urn:ngsi-ld:TrafficLightSignal:100,TrafficLightSignal,9,PRINCESA - SAN LEONARDO - PL. DE LOS CUBOS,27/02/1962,red,-3.712023,40.424761
2,urn:ngsi-ld:TrafficLightSignal:1004,TrafficLightSignal,15,TORRELAGUNA - ARTURO BALDASANO - P.P.,15/04/1981,red,-3.660855,40.455650
4,urn:ngsi-ld:TrafficLightSignal:101,TrafficLightSignal,9,PRINCESA - ROMERO ROBLEDO,16/06/1962,red,-3.718005,40.433007
5,urn:ngsi-ld:TrafficLightSignal:102,TrafficLightSignal,7,SAN BERNARDO - MAGALLANES,31/07/1962,red,-3.705185,40.431425
11,urn:ngsi-ld:TrafficLightSignal:1034,TrafficLightSignal,8,JOAQUIN LORENZO - ISLAS ALEUTIANAS,12/11/1981,red,-3.729273,40.474651
...,...,...,...,...,...,...,...,...
641,urn:ngsi-ld:TrafficLightSignal:862,TrafficLightSignal,9,FRANCOS RODRIGUEZ - PIRINEOS,01/01/1978,red,-3.715903,40.458669
642,urn:ngsi-ld:TrafficLightSignal:868,TrafficLightSignal,4,GOYA - FERNAN GONZALEZ,01/01/1978,red,-3.674138,40.424723
643,urn:ngsi-ld:TrafficLightSignal:869,TrafficLightSignal,4,GOYA - LOMBIA,01/01/1978,red,-3.672607,40.424623
644,urn:ngsi-ld:TrafficLightSignal:87,TrafficLightSignal,7,PL. EMILIO CASTELAR - PSO. CASTELLANA,01/01/1960,red,-3.688776,40.435229


# Q3: Identify all traffic lights currently in the 'Yellow' state

In [28]:
q3_df = df_tl[df_tl['status']=="yellow"]
print(f"There are a total of {len(q3_df)} lights in yellow status")
q3_df

There are a total of 66 lights in yellow status


,id,type,district,description,installationDate,status,lon,lat
7,urn:ngsi-ld:TrafficLightSignal:1021,TrafficLightSignal,7,FERNANDO CATOLICO - ARCIPRESTE HITA,08/09/1981,yellow,-3.718065,40.434301
8,urn:ngsi-ld:TrafficLightSignal:1029,TrafficLightSignal,10,CEBREROS - CARLINA - ACC. A-5,01/10/1981,yellow,-3.750700,40.403876
9,urn:ngsi-ld:TrafficLightSignal:103,TrafficLightSignal,7,AV. FILIPINAS - VALLEHERMOSO - LUCIO DEL VALLE,04/09/1962,yellow,-3.708133,40.440789
44,urn:ngsi-ld:TrafficLightSignal:1133,TrafficLightSignal,5,LOPEZ HOYOS 59,16/01/1987,yellow,-3.677058,40.442121
49,urn:ngsi-ld:TrafficLightSignal:115,TrafficLightSignal,1,BAILEN - MAYOR,02/03/1964,yellow,-3.713527,40.415094
...,...,...,...,...,...,...,...,...
604,urn:ngsi-ld:TrafficLightSignal:767,TrafficLightSignal,10,CEBREROS - CONEX. LUCERO,08/11/1977,yellow,-3.747901,40.404538
627,urn:ngsi-ld:TrafficLightSignal:835,TrafficLightSignal,8,ISLA DE AROSA - ISLAS CIES,01/01/1978,yellow,-3.716826,40.481466
632,urn:ngsi-ld:TrafficLightSignal:840,TrafficLightSignal,8,AV. CARDENAL HERRERA ORIA 279 - P.P.,01/01/1978,yellow,-3.724930,40.479343
645,urn:ngsi-ld:TrafficLightSignal:871,TrafficLightSignal,4,O'DONNELL - LOPE RUEDA,01/01/1978,yellow,-3.677511,40.421638


# Q4: List all the intersections involving ESTACION HORTALEZA that have a traffic light installed.

In [30]:
q4_df = df_tl[df_tl["description"].str.contains("HORTALEZA", case=False, na=False)]
print(f"There are a total of {len(q4_df)} entities that satisty Q4")
q4_df

There are a total of 7 entities that satisty Q4


,id,type,district,description,installationDate,status,lon,lat
92,urn:ngsi-ld:TrafficLightSignal:1299,TrafficLightSignal,16,CRTA. ESTACION HORTALEZA - VIRGEN DEL CARMEN,27/01/1992,red,-3.656241,40.473299
93,urn:ngsi-ld:TrafficLightSignal:1300,TrafficLightSignal,16,CRTA. ESTACION HORTALEZA - MONOVAR,27/01/1992,red,-3.655428,40.475118
94,urn:ngsi-ld:TrafficLightSignal:1301,TrafficLightSignal,16,CRTA. ESTACION HORTALEZA - SANTA ADELA,27/01/1992,red,-3.651966,40.479277
229,urn:ngsi-ld:TrafficLightSignal:1862,TrafficLightSignal,16,GRAN VIA DE HORTALEZA - TIJUANA - HAITI,18/12/2004,red,-3.646687,40.468384
230,urn:ngsi-ld:TrafficLightSignal:1863,TrafficLightSignal,16,GRAN VIA DE HORTALEZA - HARO,18/12/2004,green,-3.651184,40.467237
288,urn:ngsi-ld:TrafficLightSignal:2096,TrafficLightSignal,1,HORTALEZA - SERRANO ANGUITA,15/12/2009,red,-3.696794,40.426558
533,urn:ngsi-ld:TrafficLightSignal:542,TrafficLightSignal,1,HORTALEZA - MEJIA LEQUERICA,30/11/1972,red,-3.697194,40.425964


# Q5: Please identify all luminaires that are currently marked as 'broken'.

In [32]:
q5_df = df_lum[df_lum["status"]=="broken"]
print(f"There are a total of {len(q5_df)} entities that satisty Q5")
q5_df

There are a total of 69 entities that satisty Q5


,id,type,district,neighborhood,status,people_count,lon,lat
3,urn:ngsi-ld:Luminaries:230000,Luminaries,13,4,broken,4,-3.650733,40.381050
21,urn:ngsi-ld:Luminaries:230018,Luminaries,13,4,broken,0,-3.650471,40.379544
30,urn:ngsi-ld:Luminaries:230027,Luminaries,13,4,broken,1,-3.649116,40.380681
36,urn:ngsi-ld:Luminaries:230033,Luminaries,13,4,broken,1,-3.648685,40.381365
51,urn:ngsi-ld:Luminaries:230048,Luminaries,13,4,broken,1,-3.649359,40.380074
...,...,...,...,...,...,...,...,...
595,urn:ngsi-ld:Luminaries:230592,Luminaries,13,2,broken,0,-3.666175,40.388453
604,urn:ngsi-ld:Luminaries:230601,Luminaries,13,2,broken,0,-3.665924,40.388305
620,urn:ngsi-ld:Luminaries:230692,Luminaries,13,6,broken,2,-3.654925,40.397464
643,urn:ngsi-ld:Luminaries:230715,Luminaries,13,6,broken,2,-3.655600,40.397175


# ~Q6: List all luminaires where the detected affluence is greater than 7 people.~ Deprecated.

In [18]:
q6_df = df_lum[df_lum["people_count"] > 7]
q6_df

,id,type,district,neighborhood,status,people_count,lon,lat


# Q6: List all luminaires with no affluence of people.

In [33]:
q6_df = df_lum[df_lum["people_count"] == 0]
print(f"There are a total of {len(q6_df)} entities that satisfy the Q6 condition")
q6_df

There are a total of 255 entities that satisfy the Q6 condition


,id,type,district,neighborhood,status,people_count,lon,lat
4,urn:ngsi-ld:Luminaries:230001,Luminaries,13,4,on,0,-3.648982,40.380154
8,urn:ngsi-ld:Luminaries:230005,Luminaries,13,4,off,0,-3.651095,40.380453
9,urn:ngsi-ld:Luminaries:230006,Luminaries,13,4,on,0,-3.651252,40.379869
11,urn:ngsi-ld:Luminaries:230008,Luminaries,13,4,on,0,-3.649428,40.380582
12,urn:ngsi-ld:Luminaries:230009,Luminaries,13,4,on,0,-3.649610,40.380258
...,...,...,...,...,...,...,...,...
638,urn:ngsi-ld:Luminaries:230710,Luminaries,13,6,off,0,-3.654896,40.397064
639,urn:ngsi-ld:Luminaries:230711,Luminaries,13,6,on,0,-3.654502,40.396636
641,urn:ngsi-ld:Luminaries:230713,Luminaries,13,6,on,0,-3.652282,40.395662
645,urn:ngsi-ld:Luminaries:230717,Luminaries,13,6,broken,0,-3.655768,40.399573


# ~~Q7: Show me all luminaires that are of type 'LED' and are currently switched 'Off'.~~

For this questions, the idea is to test both status and technology of the luminary. However, in the JSON dataset the technology type was removed (although present in the CSV file)

# Q8: Which luminaires have an affluence of less than 5 people?

In [34]:
q8_df = df_lum[df_lum["people_count"] < 5]
print(f"There are a total of {len(q8_df)} luminaries with q8 criteria")

q8_df

There are a total of 646 luminaries with q8 criteria


,id,type,district,neighborhood,status,people_count,lon,lat
0,urn:ngsi-ld:Luminaries:148,Luminaries,16,4,on,1,-3.643609,40.473651
1,urn:ngsi-ld:Luminaries:149,Luminaries,16,4,on,1,-3.643704,40.473522
2,urn:ngsi-ld:Luminaries:150,Luminaries,16,4,on,1,-3.643525,40.473762
3,urn:ngsi-ld:Luminaries:230000,Luminaries,13,4,broken,4,-3.650733,40.381050
4,urn:ngsi-ld:Luminaries:230001,Luminaries,13,4,on,0,-3.648982,40.380154
...,...,...,...,...,...,...,...,...
645,urn:ngsi-ld:Luminaries:230717,Luminaries,13,6,broken,0,-3.655768,40.399573
646,urn:ngsi-ld:Luminaries:230718,Luminaries,13,6,on,2,-3.655179,40.399571
647,urn:ngsi-ld:Luminaries:230719,Luminaries,13,6,on,0,-3.654511,40.399487
648,urn:ngsi-ld:Luminaries:230720,Luminaries,13,6,off,1,-3.654913,40.400802


# Q9: Find all luminaires that are 'broken' and have an affluence greater than 6.

In [35]:
q9_df = df_lum[(df_lum["status"] == "broken") & (df_lum["people_count"] > 6)]
print(f"There are a total of {len(q9_df)} luminaries with q9 criteria")

q9_df

There are a total of 1 luminaries with q9 criteria


,id,type,district,neighborhood,status,people_count,lon,lat
218,urn:ngsi-ld:Luminaries:230215,Luminaries,13,5,broken,8,-3.649583,40.391908


# Q10: Are there any intersections where the traffic light is 'Green' but the nearby luminaires are 'Off' or 'Broken'?

This could not be solved by a single look-up query. For doing this, the system need somehow to cross information between the two datasets. The only common thing both have are the latitude and longitude coordinated. With that information, we need to cross data and gather all entities for a determined coordinates. For instance, using agentic behaviour and MCP tools (for the Context Broker), one possible approach could be filter all traffic lights that are green and, for each one in the result set, do the cartesian product with the luminaries that are "close" to the given GPS coordinates of this first filtering and are either "off" or "broken".

In [36]:
q10_green_df = df_tl[df_tl["status"] == "green"]
print(f"There are {len(q10_green_df)} traffic light signals in green state (first part of Q10 condition)")

There are 297 traffic light signals in green state (first part of Q10 condition)


In [39]:
q10_off_broken_df = df_lum[df_lum["status"].isin(["off", "broken"])]

print(f"There are {len(q10_off_broken_df)} luminaires that are off or broken (second part of Q10 condition)")

There are 233 luminaires that are off or broken (second part of Q10 condition)


Ideally, we should use the merged (cartesian product) dataframe with both dataset to filter that information and also filter by the calculated distance (haversine).

In [40]:
mask_q10 = (
    lum_tl_df["status_lum"].isin(["off", "broken"])
    & lum_tl_df["status_tl"].eq("green")
)

q10_green_off_broken_df = lum_tl_df[mask_q10]

#q10_green_off_broken_df = lum_tl_df[(lum_tl_df["status_lum"].isin(["off", "broken"])) & (lum_tl_df["status_tl"].isin(["green"]))]
q10_green_off_broken_df.head(3)

,id_lum,type_lum,district_lum,neighborhood,status_lum,people_count,lon_lum,lat_lum,id_tl,type_tl,district_tl,description,installationDate,status_tl,lon_tl,lat_tl,distance
298381,urn:ngsi-ld:Luminaries:230456,Luminaries,13,5,off,0,-3.651435,40.391582,urn:ngsi-ld:TrafficLightSignal:1099,TrafficLightSignal,13,AV. DE LA ALBUFERA 136 - P.P.,13/11/1984,green,-3.651377,40.391321,29.521667
171937,urn:ngsi-ld:Luminaries:230261,Luminaries,13,5,off,2,-3.646202,40.390511,urn:ngsi-ld:TrafficLightSignal:236,TrafficLightSignal,13,AV. DE LA ALBUFERA - PEDRO LABORDE,13/07/1967,green,-3.646442,40.390315,29.808693
266531,urn:ngsi-ld:Luminaries:230407,Luminaries,13,5,off,0,-3.651112,40.391527,urn:ngsi-ld:TrafficLightSignal:1099,TrafficLightSignal,13,AV. DE LA ALBUFERA 136 - P.P.,13/11/1984,green,-3.651377,40.391321,32.073942


So, the closest entities (luminaires and traffic lights) that fullfil the criteria are at 29 meters apart. 

In [41]:
print(f"The closest entities (luminaires and traffic lights) that fullfil the criteria are at {q10_green_off_broken_df.iloc[0][-1]} meters apart. ")

The closest entities (luminaires and traffic lights) that fullfil the criteria are at 29.521666698706824 meters apart. 


# ~Q11: Find traffic lights that are currently 'Red' where there is a high concentration of people (affluence > 5) waiting near the associated luminaires.~ Deprecated.

Following same reasoning that before, and using the previously calculated dataframe, we can just filter by the needed criteria:

In [80]:
near_distance = 100
mask_q11 = (
    (lum_tl_df["people_count"] > 5)
    & lum_tl_df["status_tl"].eq("red")
)

q11_red_five_people_df = lum_tl_df[mask_q11]
q11_red_five_people_df.head(3)

,id_lum,type_lum,district_lum,neighborhood,status_lum,people_count,lon_lum,lat_lum,id_tl,type_tl,district_tl,description,installationDate,status_tl,lon_tl,lat_tl,distance


so, no entities that fulfill the criteria in the merged datset

# Q11: Find traffic lights that are currently 'Red' where there is a high concentration of people (affluence < 5) waiting near the associated luminaires.

In [87]:
near_distance = 100
mask_q11 = (
    (lum_tl_df["people_count"] < 5) &
    lum_tl_df["status_tl"].eq("red") &
    lum_tl_df["distance"].lt(near_distance)
)

q11_red_five_people_df = lum_tl_df[mask_q11]
q11_red_five_people_no_dupes_df = q11_red_five_people_df[["id_tl", "description"]].drop_duplicates()

q11_red_five_people_no_dupes_df.head(10)

,id_tl,description
219879,urn:ngsi-ld:TrafficLightSignal:1649,AV. PABLO NERUDA - LUIS BUÑUEL
22816,urn:ngsi-ld:TrafficLightSignal:1212,VILLALOBOS - LEONESES
46084,urn:ngsi-ld:TrafficLightSignal:696,AV. PALOMERAS - PUERTO CARDOSO


In [88]:
print(f"There are {len(q11_red_five_people_no_dupes_df)} joint entities (lum,tl) that satisfy Q11 conditions at a distance less than {near_distance} meters")

There are 3 joint entities (lum,tl) that satisfy Q11 conditions at a distance less than 100 meters


# ~~Q12: List all active ('On') luminaires located on the same street as a 'Yellow' traffic light.~~

I think this question is near impossible to solve, or at least extremely difficult, because we need to determine the street based on the luminaire GPS coordinate, and then check if there is one traffic light on that street that satisfy the criteria. The part that looks for GPS_coord --> Street name is the hardest one.

# Q13: Identify locations where the traffic light is 'Red' AND the nearest luminaire is 'broken'.

In [44]:
mask_q13 = (
    (lum_tl_df["status_lum"].eq("broken"))
    & lum_tl_df["status_tl"].eq("red")
)

q13_red_broken_df = lum_tl_df[mask_q13]
q13_red_broken_df.head(3)

,id_lum,type_lum,district_lum,neighborhood,status_lum,people_count,lon_lum,lat_lum,id_tl,type_tl,district_tl,description,installationDate,status_tl,lon_tl,lat_tl,distance
215329,urn:ngsi-ld:Luminaries:230328,Luminaries,13,5,broken,0,-3.645669,40.388327,urn:ngsi-ld:TrafficLightSignal:1649,TrafficLightSignal,13,AV. PABLO NERUDA - LUIS BUÑUEL,10/10/2000,red,-3.645002,40.387674,91.981250
209479,urn:ngsi-ld:Luminaries:230319,Luminaries,13,5,broken,0,-3.646345,40.388017,urn:ngsi-ld:TrafficLightSignal:1649,TrafficLightSignal,13,AV. PABLO NERUDA - LUIS BUÑUEL,10/10/2000,red,-3.645002,40.387674,119.975709
208829,urn:ngsi-ld:Luminaries:230318,Luminaries,13,5,broken,1,-3.645989,40.388483,urn:ngsi-ld:TrafficLightSignal:1649,TrafficLightSignal,13,AV. PABLO NERUDA - LUIS BUÑUEL,10/10/2000,red,-3.645002,40.387674,122.795426


In [90]:
print(f"The closest location where the traffic light is RED and the nearest luminaire is broken seems to be {q13_red_broken_df.iloc[0][11]} at a distance of {q13_red_broken_df.iloc[0][-1]} meters")

The closest location where the traffic light is RED and the nearest luminaire is broken seems to be AV. PABLO NERUDA - LUIS BUÑUEL at a distance of 91.98124968700104 meters


# Q14: Find intersections where the traffic light is working (Red/Green/Yellow) but the nearby luminaires are 'Off' despite having affluence > 0.

In [48]:
mask_q14 = (
    (lum_tl_df["status_lum"].eq("off"))
    & lum_tl_df["status_tl"].isin(["red", "green", "yellow"])
    & lum_tl_df["people_count"].gt(0)
)

q14_rgy_off_df = lum_tl_df[mask_q14]
q14_rgy_off_df.head(3)

,id_lum,type_lum,district_lum,neighborhood,status_lum,people_count,lon_lum,lat_lum,id_tl,type_tl,district_tl,description,installationDate,status_tl,lon_tl,lat_tl,distance
171937,urn:ngsi-ld:Luminaries:230261,Luminaries,13,5,off,2,-3.646202,40.390511,urn:ngsi-ld:TrafficLightSignal:236,TrafficLightSignal,13,AV. DE LA ALBUFERA - PEDRO LABORDE,13/07/1967,green,-3.646442,40.390315,29.808693
360562,urn:ngsi-ld:Luminaries:230551,Luminaries,13,2,off,1,-3.669388,40.387983,urn:ngsi-ld:TrafficLightSignal:462,TrafficLightSignal,13,AV. SAN DIEGO 68 - P.P.,09/10/1971,green,-3.670013,40.388350,66.869284
230492,urn:ngsi-ld:Luminaries:230351,Luminaries,13,5,off,2,-3.645357,40.388924,urn:ngsi-ld:TrafficLightSignal:363,TrafficLightSignal,13,AV. DE LA ALBUFERA - SANTIAGO ALIO,01/07/1969,green,-3.645161,40.389523,68.571553


Answer is in the "q14_rgy_off_df" data frame. First condition is useless because it does not matter the color of the TL.

In [49]:
print(f"There are {len(q14_rgy_off_df)} joint entities (lum,tl) that satisfy Q14 conditions")

There are 62400 joint entities (lum,tl) that satisfy Q14 conditions


# Q15: Show me locations where the traffic light is NOT 'Green' AND the affluence is less than 5 OR the luminaire bulb type is 'LED'.

We don't have bulb type property (hence the check for LED). Anyway, just ignore the LED part because the condition  is an OR

In [50]:
mask_q15 = (
    lum_tl_df["status_tl"].isin(["red", "yellow"]) &
    (lum_tl_df["people_count"].lt(5))
    )
q15_notgreen_lt5 = lum_tl_df[mask_q15]
q15_notgreen_lt5.head(3)

,id_lum,type_lum,district_lum,neighborhood,status_lum,people_count,lon_lum,lat_lum,id_tl,type_tl,district_tl,description,installationDate,status_tl,lon_tl,lat_tl,distance
219879,urn:ngsi-ld:Luminaries:230335,Luminaries,13,5,off,1,-3.645700,40.388038,urn:ngsi-ld:TrafficLightSignal:1649,TrafficLightSignal,13,AV. PABLO NERUDA - LUIS BUÑUEL,10/10/2000,red,-3.645002,40.387674,71.670317
214679,urn:ngsi-ld:Luminaries:230327,Luminaries,13,5,on,0,-3.645835,40.387841,urn:ngsi-ld:TrafficLightSignal:1649,TrafficLightSignal,13,AV. PABLO NERUDA - LUIS BUÑUEL,10/10/2000,red,-3.645002,40.387674,72.941209
22816,urn:ngsi-ld:Luminaries:230032,Luminaries,13,4,on,0,-3.648367,40.381896,urn:ngsi-ld:TrafficLightSignal:1212,TrafficLightSignal,13,VILLALOBOS - LEONESES,19/07/1990,red,-3.647473,40.382007,76.761807


In [75]:
print(f"There are {len(q15_notgreen_lt5)} joint entities (lum,tl) that satisfy Q15 conditions")

There are 228038 joint entities (lum,tl) that satisfy Q15 conditions


### However, the questions is asking for locations. 
So what I'd do is to either:
- filter unique Traffic Lights that satisfy joint conditions in the cartesian product and get the intersecction streets (description field)
- filter the Luminaires that satisfy both conditions in the cartesian products and get their GPS position (lon_lum, lat_lum) (filtering duplicates GPS pairs that arise from the product with different streets).
- ✔️ filter the Luminaires that satisfy both conditions BUT in the case that ARE close -let's say t threshold meters- from a TrafficLight that satisfy the condition.

The third interpretation is considered as "most correct" one. And the threshold is set to 100 meters as "close". 

For example: Imagine a Luminaire Li with affluence 4. Then, we discover 2 StreetLights, SLj and SLk, that are a 100 meters or less from Li. Then if *at least one* of the StreetLights is not Green, then we proceed to count 1 object/event that satisfy the Question restrictions.
So, the max possible count of objects that satisfy Q15 is equal to size(luminaires) dataset

In [79]:
distance_condition = 100
mask_q15_100mAway = (
    lum_tl_df["status_tl"].isin(["red", "yellow"]) &
    (lum_tl_df["people_count"].lt(5)) &
    (lum_tl_df["distance"] <= distance_condition)
    )
q15_notgreen_lt5_lt100mts = lum_tl_df[mask_q15_100mAway]
q15_notgreen_lt5_lt100mts_unique = q15_notgreen_lt5_lt100mts[["id_tl", "description"]].drop_duplicates()
print(f"There are {len(q15_notgreen_lt5_lt100mts_unique)} places that satisfy Q15 conditions")
q15_notgreen_lt5_lt100mts_unique.head(30)


There are 3 places that satisfy Q15 conditions


,id_tl,description
219879,urn:ngsi-ld:TrafficLightSignal:1649,AV. PABLO NERUDA - LUIS BUÑUEL
22816,urn:ngsi-ld:TrafficLightSignal:1212,VILLALOBOS - LEONESES
46084,urn:ngsi-ld:TrafficLightSignal:696,AV. PALOMERAS - PUERTO CARDOSO


# ~Q16: Display entities where the traffic light is 'Yellow' OR the luminaire is 'Broken', but NOT where affluence is below 10.~ Deprecated.


In [87]:
mask_q16 = (
    (lum_tl_df["status_tl"].eq("yellow"))
    | (lum_tl_df["status_lum"].eq("broken")) & (~lum_tl_df["people_count"] < 10)
)

q16_yellow_or_brokenNotBelow10_df = lum_tl_df[mask_q16]
q16_yellow_or_brokenNotBelow10_df.head(3)

,id_lum,type_lum,district_lum,neighborhood,status_lum,people_count,lon_lum,lat_lum,id_tl,type_tl,district_tl,description,installationDate,status_tl,lon_tl,lat_tl,distance
19,urn:ngsi-ld:Luminaries:149,Luminaries,16,4,on,1,-3.643704,40.473522,urn:ngsi-ld:TrafficLightSignal:103,TrafficLightSignal,7,AV. FILIPINAS - VALLEHERMOSO - LUCIO DEL VALLE,04/09/1962,yellow,-3.708133,40.440789,6554.645809
9,urn:ngsi-ld:Luminaries:148,Luminaries,16,4,on,1,-3.643609,40.473651,urn:ngsi-ld:TrafficLightSignal:103,TrafficLightSignal,7,AV. FILIPINAS - VALLEHERMOSO - LUCIO DEL VALLE,04/09/1962,yellow,-3.708133,40.440789,6569.280967
29,urn:ngsi-ld:Luminaries:150,Luminaries,16,4,on,1,-3.643525,40.473762,urn:ngsi-ld:TrafficLightSignal:103,TrafficLightSignal,7,AV. FILIPINAS - VALLEHERMOSO - LUCIO DEL VALLE,04/09/1962,yellow,-3.708133,40.440789,6582.038597


# Q16: Display entities where the traffic light is 'Yellow' OR the luminaire is 'Broken', but NOT where affluence is below 4.

In [81]:
mask_q16 = (
    (lum_tl_df["status_tl"].eq("yellow"))
    | (lum_tl_df["status_lum"].eq("broken")) & (~lum_tl_df["people_count"] < 4)
)

q16_yellow_or_brokenNotBelow4_df = lum_tl_df[mask_q16]
q16_yellow_or_brokenNotBelow4_df.head(3)

,id_lum,type_lum,district_lum,neighborhood,status_lum,people_count,lon_lum,lat_lum,id_tl,type_tl,district_tl,description,installationDate,status_tl,lon_tl,lat_tl,distance
181687,urn:ngsi-ld:Luminaries:230276,Luminaries,13,5,broken,2,-3.646635,40.390607,urn:ngsi-ld:TrafficLightSignal:236,TrafficLightSignal,13,AV. DE LA ALBUFERA - PEDRO LABORDE,13/07/1967,green,-3.646442,40.390315,36.334877
251736,urn:ngsi-ld:Luminaries:230384,Luminaries,15,9,broken,0,-3.665121,40.478765,urn:ngsi-ld:TrafficLightSignal:1675,TrafficLightSignal,15,GOLFO DE SALONICA - DALIA,27/04/2001,green,-3.664924,40.478393,44.617054
187537,urn:ngsi-ld:Luminaries:230285,Luminaries,13,5,broken,4,-3.646870,40.390641,urn:ngsi-ld:TrafficLightSignal:236,TrafficLightSignal,13,AV. DE LA ALBUFERA - PEDRO LABORDE,13/07/1967,green,-3.646442,40.390315,51.271413


In [53]:
print(f"There are {len(q16_yellow_or_brokenNotBelow4_df)} joint entities (lum,tl) that satisfy Q16 conditions")

There are 83196 joint entities (lum,tl) that satisfy Q16 conditions


# Q17: List all intersections where the traffic light is neither 'Red' nor 'Green'.

In [60]:
mask_q17 = (
    ~(df_tl["status"].eq("red")) &
    ~(df_tl["status"].eq("green"))
)

q17_neitherRedNorGreen_df = df_tl[mask_q17]
q17_neitherRedNorGreen_df.head(3)

,id,type,district,description,installationDate,status,lon,lat
7,urn:ngsi-ld:TrafficLightSignal:1021,TrafficLightSignal,7,FERNANDO CATOLICO - ARCIPRESTE HITA,08/09/1981,yellow,-3.718065,40.434301
8,urn:ngsi-ld:TrafficLightSignal:1029,TrafficLightSignal,10,CEBREROS - CARLINA - ACC. A-5,01/10/1981,yellow,-3.750700,40.403876
9,urn:ngsi-ld:TrafficLightSignal:103,TrafficLightSignal,7,AV. FILIPINAS - VALLEHERMOSO - LUCIO DEL VALLE,04/09/1962,yellow,-3.708133,40.440789


In [64]:
print(f"There are {len(q17_neitherRedNorGreen_df)} joint entities (lum,tl) that satisfy Q17 conditions")

There are 66 joint entities (lum,tl) that satisfy Q17 conditions


# Q18: Find pairs of Traffic Lights and Luminaires where the luminaire affluence is high (>10) but the traffic light is 'Green' (allowing flow).

In [62]:
mask_q18 = (
    (lum_tl_df["status_tl"].eq("green"))
    & (lum_tl_df["people_count"] > 10)
)

q18_highflux_green_df = lum_tl_df[mask_q18]
q18_highflux_green_df.head(3)

,id_lum,type_lum,district_lum,neighborhood,status_lum,people_count,lon_lum,lat_lum,id_tl,type_tl,district_tl,description,installationDate,status_tl,lon_tl,lat_tl,distance


In [65]:
print(f"There are {len(q18_highflux_green_df)} joint entities (lum,tl) that satisfy Q18 conditions")

There are 0 joint entities (lum,tl) that satisfy Q18 conditions


# Q19: Based on the data, create a list of locations that require immediate repair (broken luminaires).

In [68]:
mask_q19 = (
    (df_lum["status"].eq("broken"))
)

q19_brokenlum_df = df_lum[mask_q19]
q19_brokenlum_df.head(650)

,id,type,district,neighborhood,status,people_count,lon,lat
3,urn:ngsi-ld:Luminaries:230000,Luminaries,13,4,broken,4,-3.650733,40.381050
21,urn:ngsi-ld:Luminaries:230018,Luminaries,13,4,broken,0,-3.650471,40.379544
30,urn:ngsi-ld:Luminaries:230027,Luminaries,13,4,broken,1,-3.649116,40.380681
36,urn:ngsi-ld:Luminaries:230033,Luminaries,13,4,broken,1,-3.648685,40.381365
51,urn:ngsi-ld:Luminaries:230048,Luminaries,13,4,broken,1,-3.649359,40.380074
...,...,...,...,...,...,...,...,...
595,urn:ngsi-ld:Luminaries:230592,Luminaries,13,2,broken,0,-3.666175,40.388453
604,urn:ngsi-ld:Luminaries:230601,Luminaries,13,2,broken,0,-3.665924,40.388305
620,urn:ngsi-ld:Luminaries:230692,Luminaries,13,6,broken,2,-3.654925,40.397464
643,urn:ngsi-ld:Luminaries:230715,Luminaries,13,6,broken,2,-3.655600,40.397175


In [67]:
print(f"There are {len(q19_brokenlum_df)} luminaires that satisfy Q19 conditions")

There are 69 luminaires that satisfy Q19 conditions


# Q20: Compare the number of 'Red' traffic lights to the number of 'Broken' luminaires in the provided area.

In [70]:
mask_q20_lum = (
    (df_lum["status"].eq("broken"))
)
mask_q20_tl = (
    (df_tl["status"].eq("red"))
)

q20_broken_df = df_lum[mask_q20_lum]
q20_red_df = df_tl[mask_q20_tl]

In [72]:
q20_red_df.head(650)
print(f"number of red traffic lights: {len(q20_red_df)}")

number of red traffic lights: 287


In [74]:
q20_broken_df.head(650)
print(f"number of broken lum: {len(q20_broken_df)}")

number of broken lum: 69
